# Metorial + LangChain Example

This notebook demonstrates how to use Metorial tools with LangChain agents.

In [ ]:
# Install dependencies
%pip install metorial langchain langchain-anthropic python-dotenv

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

# Make sure these environment variables are set
assert os.getenv("METORIAL_API_KEY"), "Set METORIAL_API_KEY"
assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY"
assert os.getenv("EXA_DEPLOYMENT_ID"), "Set EXA_DEPLOYMENT_ID"

In [ ]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate

from metorial import Metorial
from metorial.integrations.langchain import create_langchain_tools

In [ ]:
metorial = Metorial(api_key=os.getenv("METORIAL_API_KEY"))

In [ ]:
async def run_agent(query: str):
  async with metorial.provider_session(
    provider="anthropic",
    server_deployments=[os.getenv("EXA_DEPLOYMENT_ID")],
  ) as session:
    tools = create_langchain_tools(session)

    print(f"Available tools: {[t.name for t in tools]}")

    llm = ChatAnthropic(model="claude-sonnet-4-20250514")
    prompt = ChatPromptTemplate.from_messages(
      [
        ("system", "You are a helpful research assistant."),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
      ]
    )

    agent = create_tool_calling_agent(llm, tools, prompt)
    executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

    result = await executor.ainvoke({"input": query})
    return result["output"]

In [ ]:
result = await run_agent("Search for the latest Python 3.13 features")
print(result)